# GCE Eligibility - 02: Baseline Models

Trains logistic regression and decision tree baselines on the preprocessed splits from `01_data_prep`.

In [ ]:
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score

SPLITS = "/Volumes/ml_final_workspace/default/ml_final/splits"

X_train = np.load(f"{SPLITS}/X_train.npy")
X_val   = np.load(f"{SPLITS}/X_val.npy")
X_test  = np.load(f"{SPLITS}/X_test.npy")
y_train = np.load(f"{SPLITS}/y_train.npy")
y_val   = np.load(f"{SPLITS}/y_val.npy")
y_test  = np.load(f"{SPLITS}/y_test.npy")

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Positive rate — train: {y_train.mean():.4f}, val: {y_val.mean():.4f}, test: {y_test.mean():.4f}")


In [ ]:
mlflow.set_experiment("/gce_eligibility")

# logistic regression
with mlflow.start_run(run_name="logistic_regression"):
    params = {"class_weight": "balanced", "max_iter": 1000, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params)
    lr.fit(X_train, y_train)
    lr_preds = lr.predict(X_test)

    lr_acc  = accuracy_score(y_test, lr_preds)
    lr_f1   = f1_score(y_test, lr_preds, zero_division=0)
    lr_prec = precision_score(y_test, lr_preds, zero_division=0)
    lr_rec  = recall_score(y_test, lr_preds, zero_division=0)

    mlflow.log_metrics({"accuracy": lr_acc, "f1": lr_f1, "precision": lr_prec, "recall": lr_rec})
    mlflow.sklearn.log_model(lr, "model")

    print("LOGISTIC REGRESSION")
    print(f"  Accuracy:  {lr_acc:.4f}")
    print(f"  F1:        {lr_f1:.4f}")
    print(f"  Precision: {lr_prec:.4f}")
    print(f"  Recall:    {lr_rec:.4f}")


In [ ]:
# decision tree
with mlflow.start_run(run_name="decision_tree"):
    params = {"class_weight": "balanced", "max_depth": 10, "random_state": 42}
    mlflow.log_params(params)

    dt = DecisionTreeClassifier(**params)
    dt.fit(X_train, y_train)
    dt_preds = dt.predict(X_test)

    dt_acc  = accuracy_score(y_test, dt_preds)
    dt_f1   = f1_score(y_test, dt_preds, zero_division=0)
    dt_prec = precision_score(y_test, dt_preds, zero_division=0)
    dt_rec  = recall_score(y_test, dt_preds, zero_division=0)

    mlflow.log_metrics({"accuracy": dt_acc, "f1": dt_f1, "precision": dt_prec, "recall": dt_rec})
    mlflow.sklearn.log_model(dt, "model")

    print("DECISION TREE")
    print(f"  Accuracy:  {dt_acc:.4f}")
    print(f"  F1:        {dt_f1:.4f}")
    print(f"  Precision: {dt_prec:.4f}")
    print(f"  Recall:    {dt_rec:.4f}")


In [ ]:
print(f"\n{'Model':<25} {'Accuracy':<12} {'F1':<10} {'Precision':<12} {'Recall':<10}")
print("-" * 69)
print(f"{'Logistic Regression':<25} {lr_acc:<12.4f} {lr_f1:<10.4f} {lr_prec:<12.4f} {lr_rec:<10.4f}")
print(f"{'Decision Tree':<25} {dt_acc:<12.4f} {dt_f1:<10.4f} {dt_prec:<12.4f} {dt_rec:<10.4f}")
print(f"{'MLP (Local, RTX 3060Ti)':<25} {'0.9875':<12} {'0.8812':<10} {'0.8182':<12} {'0.9532':<10}")
print("\nOpen the MLflow Experiments tab to compare runs visually.")
